## Exploratory Data Analysis and Cleanup for Children's Books

In [1]:
# importing libraries
import pandas as pd
pd.set_option("display.max_columns", None)
import os
from dotenv import load_dotenv
import numpy as np
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ast
from collections import OrderedDict
import re

/Users/navyajain/Downloads/build/agentic-book-recommendation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DetectorFactory.seed = 42

In [3]:
# paths
load_dotenv()
#children_path = os.getenv("CHILDREN_BOOKS")

True

In [4]:
#print(children_path)

In [5]:
comic_df = pd.read_json('../data/books/goodreads_books_young_adult.json', lines=True)

In [6]:
comic_df.shape

(93398, 29)

In [7]:
comic_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,,1,[147734],US,,"[{'count': '1057', 'name': 'to-read'}, {'count...",B0056A00P4,true,4.04,B0056A00P4,"[519546, 1295074, 21407416]",This is the final tale in the bestselling auth...,,https://www.goodreads.com/book/show/12182387-t...,"[{'author_id': '50873', 'role': ''}, {'author_...",,,,,,,,https://www.goodreads.com/book/show/12182387-t...,https://s.gr-assets.com/assets/nophoto/book/11...,12182387,4,285263,"The Passion (Dark Visions, #3)","The Passion (Dark Visions, #3)"
1,,2,[425995],US,,"[{'count': '1010', 'name': 'to-read'}, {'count...",B006KLYIAG,true,3.80,B006KLYIAG,"[13400912, 13327517, 18107102, 15797097, 11472...",Life should be simple for Cassie.\nFor the sma...,,https://www.goodreads.com/book/show/20135365-h...,"[{'author_id': '5395324', 'role': ''}]",,,,,,,,https://www.goodreads.com/book/show/20135365-h...,https://s.gr-assets.com/assets/nophoto/book/11...,20135365,5,18450480,Hope's Daughter,Hope's Daughter
2,0698143760,17,[493993],US,,"[{'count': '1799', 'name': 'fantasy'}, {'count...",,true,3.80,,"[15728807, 17182499, 15673520, 16081758, 17842...",Wanted by no one.\nHunted by everyone.\nSixtee...,ebook,https://www.goodreads.com/book/show/21401181-h...,"[{'author_id': '7314532', 'role': ''}]",Viking Children's,416,4,9780698143760,3,,2014,https://www.goodreads.com/book/show/21401181-h...,https://images.gr-assets.com/books/1394747643m...,21401181,33,24802827,"Half Bad (Half Life, #1)","Half Bad (Half Life, #1)"
3,,9,[176160],US,eng,"[{'count': '7173', 'name': 'to-read'}, {'count...",B0042JSOQC,true,4.35,B004IYJDXY,"[25861113, 7430195, 18765937, 6120544, 3247550...",It all comes down to this.\nVlad's running out...,,https://www.goodreads.com/book/show/10099492-t...,"[{'author_id': '293603', 'role': ''}]",,,,,,,,https://www.goodreads.com/book/show/10099492-t...,https://s.gr-assets.com/assets/nophoto/book/11...,10099492,152,10800440,Twelfth Grade Kills (The Chronicles of Vladimi...,Twelfth Grade Kills (The Chronicles of Vladimi...
4,0990662616,428,[],US,eng,"[{'count': '9481', 'name': 'to-read'}, {'count...",,false,3.71,B00MW0MTGE,"[20499652, 17934493, 13518102, 16210411, 17149...",The future world is at peace.\nElla Shepherd h...,Paperback,https://www.goodreads.com/book/show/22642971-t...,"[{'author_id': '4018722', 'role': ''}]",Scripturient Books,351,6,9780990662617,10,Special Edition,2014,https://www.goodreads.com/book/show/22642971-t...,https://images.gr-assets.com/books/1406979059m...,22642971,1525,42144295,The Body Electric,The Body Electric


In [8]:
df = comic_df.copy()

| Column | Meaning |
|---|---|
| `isbn` | 10-digit ISBN for the book edition, if available. |
| `text_reviews_count` | Number of written/text reviews for this book edition. |
| `series` | List of Goodreads series IDs the book belongs to. Empty list means no series or unavailable. |
| `country_code` | Country code from Goodreads metadata, often `US`. |
| `language_code` | Language code for the edition, such as `eng`, `en-US`, etc. Can be blank. |
| `popular_shelves` | User-created Goodreads shelf tags for the book, with shelf name and count. |
| `asin` | Amazon Standard Identification Number, if available. |
| `is_ebook` | Whether the edition is an ebook, stored as `"true"` or `"false"`. |
| `average_rating` | Average Goodreads rating for the book, usually from 1 to 5. |
| `kindle_asin` | Kindle-specific Amazon ID, if available. |
| `similar_books` | List of Goodreads `book_id`s marked as similar books. |
| `description` | Book description or blurb text. Can be empty. |
| `format` | Book format, such as `Hardcover`, `Paperback`, or `Kindle Edition`. |
| `link` | Goodreads link for the book page. |
| `authors` | List of author objects, usually containing `author_id` and `role`. |
| `publisher` | Publisher name for this edition. |
| `num_pages` | Number of pages, if available. |
| `publication_day` | Publication day of the month, if available. |
| `isbn13` | 13-digit ISBN for the book edition, if available. |
| `publication_month` | Publication month, if available. |
| `edition_information` | Extra edition information, such as `Anniversary Edition` or `First Edition`. |
| `publication_year` | Publication year for this edition. |
| `url` | Goodreads URL for the book page. Usually similar to `link`. |
| `image_url` | URL for the book cover image. |
| `book_id` | Goodreads ID for this specific book edition. Useful for joining with reviews/interactions. |
| `ratings_count` | Number of ratings for this book edition. |
| `work_id` | Goodreads work ID. Groups multiple editions of the same underlying book. Useful for deduplication. |
| `title` | Full book title as shown on Goodreads, often including series information. |
| `title_without_series` | Book title with series information removed. Better for clean display/search. |

In [9]:
df.shape

(93398, 29)

In [10]:
df.columns

Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin',
       'similar_books', 'description', 'format', 'link', 'authors',
       'publisher', 'num_pages', 'publication_day', 'isbn13',
       'publication_month', 'edition_information', 'publication_year', 'url',
       'image_url', 'book_id', 'ratings_count', 'work_id', 'title',
       'title_without_series'],
      dtype='str')

In [11]:
df = df[['title', 'title_without_series', 'series', 'country_code', 'language_code',
       'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id', 'book_id',
       'text_reviews_count', 'average_rating', 'ratings_count', 'popular_shelves',
       'is_ebook', 'format', 'num_pages', 'publisher', 'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [12]:
df.shape

(93398, 26)

In [13]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,"The Passion (Dark Visions, #3)","The Passion (Dark Visions, #3)",[147734],US,,,B0056A00P4,B0056A00P4,,285263,12182387,1,4.04,4,"[{'count': '1057', 'name': 'to-read'}, {'count...",true,,,,,,,,This is the final tale in the bestselling auth...,"[{'author_id': '50873', 'role': ''}, {'author_...","[519546, 1295074, 21407416]"
1,Hope's Daughter,Hope's Daughter,[425995],US,,,B006KLYIAG,B006KLYIAG,,18450480,20135365,2,3.80,5,"[{'count': '1010', 'name': 'to-read'}, {'count...",true,,,,,,,,Life should be simple for Cassie.\nFor the sma...,"[{'author_id': '5395324', 'role': ''}]","[13400912, 13327517, 18107102, 15797097, 11472..."
2,"Half Bad (Half Life, #1)","Half Bad (Half Life, #1)",[493993],US,,0698143760,,,9780698143760,24802827,21401181,17,3.80,33,"[{'count': '1799', 'name': 'fantasy'}, {'count...",true,ebook,416,Viking Children's,4,3,2014,,Wanted by no one.\nHunted by everyone.\nSixtee...,"[{'author_id': '7314532', 'role': ''}]","[15728807, 17182499, 15673520, 16081758, 17842..."
3,Twelfth Grade Kills (The Chronicles of Vladimi...,Twelfth Grade Kills (The Chronicles of Vladimi...,[176160],US,eng,,B0042JSOQC,B004IYJDXY,,10800440,10099492,9,4.35,152,"[{'count': '7173', 'name': 'to-read'}, {'count...",true,,,,,,,,It all comes down to this.\nVlad's running out...,"[{'author_id': '293603', 'role': ''}]","[25861113, 7430195, 18765937, 6120544, 3247550..."
4,The Body Electric,The Body Electric,[],US,eng,0990662616,,B00MW0MTGE,9780990662617,42144295,22642971,428,3.71,1525,"[{'count': '9481', 'name': 'to-read'}, {'count...",false,Paperback,351,Scripturient Books,6,10,2014,Special Edition,The future world is at peace.\nElla Shepherd h...,"[{'author_id': '4018722', 'role': ''}]","[20499652, 17934493, 13518102, 16210411, 17149..."


In [14]:
df['clean_title'] = df['title_without_series']

In [15]:
df.shape

(93398, 27)

### Detecting languages

In [16]:
def detect_lang(text):
    if pd.isna(text) or len(str(text).strip()) < 30:
        return np.nan

    try:
        lang = detect(str(text))
        return "eng" if lang == "en" else lang
    except:
        return np.nan

In [17]:
# normalize empty strings to NaN
df["language_code_final"] = df["language_code"].replace("", np.nan)

# create title + description text
df["language_text"] = (
    df["clean_title"].fillna("") + " " + df["description"].fillna("")
)

# detect only missing language codes
missing_lang_mask = df["language_code_final"].isna()

texts_to_detect = df.loc[missing_lang_mask, "language_text"]

detected_languages = [
    detect_lang(text)
    for text in tqdm(texts_to_detect, desc="Detecting languages")
]

df.loc[missing_lang_mask, "detected_language_code"] = detected_languages

# fill missing language code with detected language
df["language_code_final"] = df["language_code_final"].fillna(
    df["detected_language_code"]
)

# normalize English variants
df["language_code_final"] = (
    df["language_code_final"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "en": "eng",
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)

Detecting languages: 100%|██████████| 29632/29632 [01:25<00:00, 348.10it/s]


In [18]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,clean_title,language_code_final,language_text,detected_language_code
0,"The Passion (Dark Visions, #3)","The Passion (Dark Visions, #3)",[147734],US,,,B0056A00P4,B0056A00P4,,285263,12182387,1,4.04,4,"[{'count': '1057', 'name': 'to-read'}, {'count...",true,,,,,,,,This is the final tale in the bestselling auth...,"[{'author_id': '50873', 'role': ''}, {'author_...","[519546, 1295074, 21407416]","The Passion (Dark Visions, #3)",eng,"The Passion (Dark Visions, #3) This is the fin...",eng
1,Hope's Daughter,Hope's Daughter,[425995],US,,,B006KLYIAG,B006KLYIAG,,18450480,20135365,2,3.80,5,"[{'count': '1010', 'name': 'to-read'}, {'count...",true,,,,,,,,Life should be simple for Cassie.\nFor the sma...,"[{'author_id': '5395324', 'role': ''}]","[13400912, 13327517, 18107102, 15797097, 11472...",Hope's Daughter,eng,Hope's Daughter Life should be simple for Cass...,eng
2,"Half Bad (Half Life, #1)","Half Bad (Half Life, #1)",[493993],US,,0698143760,,,9780698143760,24802827,21401181,17,3.80,33,"[{'count': '1799', 'name': 'fantasy'}, {'count...",true,ebook,416,Viking Children's,4,3,2014,,Wanted by no one.\nHunted by everyone.\nSixtee...,"[{'author_id': '7314532', 'role': ''}]","[15728807, 17182499, 15673520, 16081758, 17842...","Half Bad (Half Life, #1)",eng,"Half Bad (Half Life, #1) Wanted by no one.\nHu...",eng
3,Twelfth Grade Kills (The Chronicles of Vladimi...,Twelfth Grade Kills (The Chronicles of Vladimi...,[176160],US,eng,,B0042JSOQC,B004IYJDXY,,10800440,10099492,9,4.35,152,"[{'count': '7173', 'name': 'to-read'}, {'count...",true,,,,,,,,It all comes down to this.\nVlad's running out...,"[{'author_id': '293603', 'role': ''}]","[25861113, 7430195, 18765937, 6120544, 3247550...",Twelfth Grade Kills (The Chronicles of Vladimi...,eng,Twelfth Grade Kills (The Chronicles of Vladimi...,nan
4,The Body Electric,The Body Electric,[],US,eng,0990662616,,B00MW0MTGE,9780990662617,42144295,22642971,428,3.71,1525,"[{'count': '9481', 'name': 'to-read'}, {'count...",false,Paperback,351,Scripturient Books,6,10,2014,Special Edition,The future world is at peace.\nElla Shepherd h...,"[{'author_id': '4018722', 'role': ''}]","[20499652, 17934493, 13518102, 16210411, 17149...",The Body Electric,eng,The Body Electric The future world is at peace...,nan


In [19]:
df.columns

Index(['title', 'title_without_series', 'series', 'country_code',
       'language_code', 'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id',
       'book_id', 'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'is_ebook', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'clean_title', 'language_code_final', 'language_text',
       'detected_language_code'],
      dtype='str')

In [20]:
df.shape

(93398, 30)

In [21]:
df=df[['clean_title', 'series', 'language_code_final', 
       'isbn', 'isbn13', 'work_id', 'book_id', 
       'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [22]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,"The Passion (Dark Visions, #3)",[147734],eng,,,285263,12182387,1,4.04,4,"[{'count': '1057', 'name': 'to-read'}, {'count...",,,,,,,,This is the final tale in the bestselling auth...,"[{'author_id': '50873', 'role': ''}, {'author_...","[519546, 1295074, 21407416]"
1,Hope's Daughter,[425995],eng,,,18450480,20135365,2,3.80,5,"[{'count': '1010', 'name': 'to-read'}, {'count...",,,,,,,,Life should be simple for Cassie.\nFor the sma...,"[{'author_id': '5395324', 'role': ''}]","[13400912, 13327517, 18107102, 15797097, 11472..."
2,"Half Bad (Half Life, #1)",[493993],eng,0698143760,9780698143760,24802827,21401181,17,3.80,33,"[{'count': '1799', 'name': 'fantasy'}, {'count...",ebook,416,Viking Children's,4,3,2014,,Wanted by no one.\nHunted by everyone.\nSixtee...,"[{'author_id': '7314532', 'role': ''}]","[15728807, 17182499, 15673520, 16081758, 17842..."
3,Twelfth Grade Kills (The Chronicles of Vladimi...,[176160],eng,,,10800440,10099492,9,4.35,152,"[{'count': '7173', 'name': 'to-read'}, {'count...",,,,,,,,It all comes down to this.\nVlad's running out...,"[{'author_id': '293603', 'role': ''}]","[25861113, 7430195, 18765937, 6120544, 3247550..."
4,The Body Electric,[],eng,0990662616,9780990662617,42144295,22642971,428,3.71,1525,"[{'count': '9481', 'name': 'to-read'}, {'count...",Paperback,351,Scripturient Books,6,10,2014,Special Edition,The future world is at peace.\nElla Shepherd h...,"[{'author_id': '4018722', 'role': ''}]","[20499652, 17934493, 13518102, 16210411, 17149..."


In [23]:
df.shape

(93398, 21)

In [24]:
df = df[
    df["language_code_final"].astype("string").str.strip().str.lower() == "eng"
].copy()

In [25]:
df.shape

(71383, 21)

In [26]:
df['format'].value_counts().shape

(115,)

In [27]:
unique_formats = df["format"].dropna().unique().tolist()
print(unique_formats)

['', 'ebook', 'Paperback', 'Hardcover', 'Kindle Edition', 'Audio', 'Mass Market Paperback', 'Audio CD', 'Audiobook', 'Audible Audio', 'Audio Cassette', 'Library Binding', 'Preloaded Digital Audio Player', 'Audiobook, Unabridged Audiobook (11hrs 41min)', 'Nook', "Audio CD's", 'Audiobook, Unabridged Audiobook (8hrs 55min)', 'hardcover', 'Unknown Binding', 'MP3 CD', 'MP3 Book', 'Paperback, ebook', 'audio', 'paperback and ebook', 'Online Fiction', 'paperback', 'Advanced Review Copy', 'Newsletter Serial/Wattpad Story', 'P.D.F.', 'Uncorrected Proof, paperback', 'Kobo ebook', 'Wattpad', 'Digital Audio (Unabridged)', 'Softcover', 'Broche', 'Paperback, Kindle, Ebook, Audio', 'Boxed Set', 'online fiction', 'free podcast novel', 'Audiobook, Unabridged Audiobook (17hrs 37min)', 'Epub Edition', 'Free Online Read', 'Audiobook, Unabridged Audiobook (16hrs 19min)', 'Kindle', 'Online Publication', 'Trade Paperback', 'softcover', 'Playaway', 'iBooks', 'epub', 'Hardback', 'Poche', 'hardback', 'Board Book

In [28]:
# 1. Define readable formats to keep
keep_formats = {
    # Standard print
    "Paperback",
    "paperback",
    "Paper back",
    "PB",
    "Hardcover",
    "hardcover",
    "Hardback",
    "hardback",
    "Hardcover - Large Print",
    "Mass Market Paperback",
    "Trade Paperback",
    "Trade paperback",
    "B Trade paperback",
    "Export Trade Paperback",
    "Paperback - C Format",
    "Softcover",
    "softcover",
    "Perfect Paperback",
    "Library Binding",
    "School &amp; Library Binding",
    "Turtleback",
    "Spiral-bound",
    "Unbound",
    "Leather Bound",
    "Board Book",
    "Board book",
    "Newsprint",

    # ARCs / proofs / review copies
    "Advanced Review Copy",
    "Advanced Reader Copy",
    "Advance Reader's Copy",
    "Reader's Advance Copy",
    "ARC",
    "ARC E-book",
    "Galley Proof",
    "Uncorrected Proof, paperback",

    # Ebooks / digital readable text
    "ebook",
    "Ebook",
    "e-book",
    "Kindle Edition",
    "Kindle",
    "Kindle Fire",
    "Nook",
    "Nook eBook",
    "Nook Book",
    "Nook book",
    "Nook ebook",
    "Nook Edition",
    "NOOKbook/ BN ebook",
    "Kobo ebook",
    "iBooks",
    "Scribd",
    "epub",
    "Epub Edition",
    "P.D.F.",
    "PDF",
    "Webpage",

    # Mixed print + ebook
    "Paperback, ebook",
    "paperback and ebook",
    "Paperback and eBook",
    "Paperback, eBook",
    "ebook, paperback",
    "Paperback, Kindle, Ebook, Audio",  # keep because it includes readable formats
    "Hardcover, ebook",

    # Online / web readable text
    "Online",
    "Online Fiction",
    "online fiction",
    "Online Fiction - Complete",
    "Online Publication",
    "Online Serial",
    "Free Online Fiction",
    "Free Online Read",
    "Free short story download",
    "Wattpad",
    "Newsletter Serial/Wattpad Story",
    "Short Story",

    # Box/set/special readable books
    "Boxed Set",
    "Hardcover book-plus product",
    "Hardbound book with creepy hand lock and key",
    "Chaptered, Paperback.",

    # Foreign labels that are readable print
    "Broche",
    "broche",
    "Poche",
    "Relie",
}

drop_formats = {
    # Missing / unknown
    "",
    "Unknown Binding",

    # Audio
    "Audio",
    "audio",
    "Audio CD",
    "Audio CD's",
    "Audio Cassette",
    "Audiobook",
    "Audible Audio",
    "Audiobook, Unabridged Audiobook (11hrs 41min)",
    "Audiobook, Unabridged Audiobook (8hrs 55min)",
    "Audiobook, Unabridged Audiobook (17hrs 37min)",
    "Audiobook, Unabridged Audiobook (16hrs 19min)",
    "Audiobook, Unabridged Audiobook (1hrs 31min)",
    "Audiobook, Unabridged Audiobook (15hrs 31min)",
    "Audiobook, Unabridged Audiobook (1hrs 58min)",
    "Audiobook, Unabridged Audiobook (14hrs 40min)",
    "Audiobook, Unabridged Audiobook (17hrs 9min)",
    "Digital Audio",
    "Digital Audio (Unabridged)",
    "Downloadable Audiobook",
    "MP3 CD",
    "MP3 Book",
    "mp3 Audiobook",
    "audiobook mp3",
    "OverDrive MP3 Audiobook",
    "Preloaded Digital Audio Player",
    "Pre-recorded MP3 player",
    "Playaway",
    "Podiobook",
    "free podcast novel",

    # Mixed / non-readable product
    "Book and Toy",
}

# 4. Clean the format column
df["format_clean"] = (df["format"].fillna("").astype(str).str.strip())

# 5. Assign each row to a format group
df["format_group"] = np.select(
    [
        df["format_clean"].isin(keep_formats),
        df["format_clean"].isin(drop_formats)
    ],
    ["text", "other"], default="review")

# 6. Check any unclassified formats
df["format_group"].value_counts()

format_group
text     51826
other    19557
Name: count, dtype: int64

In [29]:
df = df[df["format_group"] == "text"].copy()

In [30]:
df["format_group"].value_counts()

format_group
text    51826
Name: count, dtype: int64

In [31]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,format_clean,format_group
2,"Half Bad (Half Life, #1)",[493993],eng,0698143760,9780698143760,24802827,21401181,17,3.80,33,"[{'count': '1799', 'name': 'fantasy'}, {'count...",ebook,416,Viking Children's,4,3,2014,,Wanted by no one.\nHunted by everyone.\nSixtee...,"[{'author_id': '7314532', 'role': ''}]","[15728807, 17182499, 15673520, 16081758, 17842...",ebook,text
4,The Body Electric,[],eng,0990662616,9780990662617,42144295,22642971,428,3.71,1525,"[{'count': '9481', 'name': 'to-read'}, {'count...",Paperback,351,Scripturient Books,6,10,2014,Special Edition,The future world is at peace.\nElla Shepherd h...,"[{'author_id': '4018722', 'role': ''}]","[20499652, 17934493, 13518102, 16210411, 17149...",Paperback,text
5,Like Water,[],eng,0062373374,9780062373373,52237886,31556136,45,3.89,109,"[{'count': '3010', 'name': 'to-read'}, {'count...",Hardcover,304,Balzer + Bray,17,10,2017,,A gorgeously written and deeply felt literary ...,"[{'author_id': '6537142', 'role': ''}]",[],Hardcover,text
9,The Ticket,[],eng,1941103863,9781941103869,45179443,25421507,15,3.82,23,"[{'count': '140', 'name': 'to-read'}, {'count'...",Paperback,210,Firefly Southern Fiction,20,5,2015,,Librarian Note: See Alternate Cover Edition ....,"[{'author_id': '539783', 'role': ''}]",[],Paperback,text
10,"Out of Order (The Survivor's Club, #1)",[926363],eng,1634760123,9781634760126,45171289,25414982,31,3.85,61,"[{'count': '366', 'name': 'to-read'}, {'count'...",ebook,180,Harmony Ink Press,21,3,2015,,"Corinna ""Corey"" Nguyen's life seems perfectly ...","[{'author_id': '13850345', 'role': ''}]","[25375979, 24740779, 32320630, 25005560, 23129...",ebook,text


In [32]:
df.shape

(51826, 23)

In [33]:
df.columns

Index(['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'format_clean', 'format_group'],
      dtype='str')

In [34]:
df = df[['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format_clean', 'num_pages', 'publisher',
       'publication_year', 'description', 'authors', 'similar_books']]

In [35]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format_clean,num_pages,publisher,publication_year,description,authors,similar_books
2,"Half Bad (Half Life, #1)",[493993],eng,0698143760,9780698143760,24802827,21401181,17,3.80,33,"[{'count': '1799', 'name': 'fantasy'}, {'count...",ebook,416,Viking Children's,2014,Wanted by no one.\nHunted by everyone.\nSixtee...,"[{'author_id': '7314532', 'role': ''}]","[15728807, 17182499, 15673520, 16081758, 17842..."
4,The Body Electric,[],eng,0990662616,9780990662617,42144295,22642971,428,3.71,1525,"[{'count': '9481', 'name': 'to-read'}, {'count...",Paperback,351,Scripturient Books,2014,The future world is at peace.\nElla Shepherd h...,"[{'author_id': '4018722', 'role': ''}]","[20499652, 17934493, 13518102, 16210411, 17149..."
5,Like Water,[],eng,0062373374,9780062373373,52237886,31556136,45,3.89,109,"[{'count': '3010', 'name': 'to-read'}, {'count...",Hardcover,304,Balzer + Bray,2017,A gorgeously written and deeply felt literary ...,"[{'author_id': '6537142', 'role': ''}]",[]
9,The Ticket,[],eng,1941103863,9781941103869,45179443,25421507,15,3.82,23,"[{'count': '140', 'name': 'to-read'}, {'count'...",Paperback,210,Firefly Southern Fiction,2015,Librarian Note: See Alternate Cover Edition ....,"[{'author_id': '539783', 'role': ''}]",[]
10,"Out of Order (The Survivor's Club, #1)",[926363],eng,1634760123,9781634760126,45171289,25414982,31,3.85,61,"[{'count': '366', 'name': 'to-read'}, {'count'...",ebook,180,Harmony Ink Press,2015,"Corinna ""Corey"" Nguyen's life seems perfectly ...","[{'author_id': '13850345', 'role': ''}]","[25375979, 24740779, 32320630, 25005560, 23129..."


In [36]:
def clean_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip() == "":
        return np.nan
    if isinstance(x, list) and len(x) == 0:
        return np.nan
    return x


def unique_list(series):
    seen = set()
    result = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        key = str(x)

        if key not in seen:
            seen.add(key)
            result.append(x)

    return result


def first_non_missing(series):
    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        return x

    return np.nan


def longest_text(series):
    texts = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, str) and x.strip() != "":
            texts.append(x.strip())

    if len(texts) == 0:
        return np.nan

    return max(texts, key=len)


def max_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.max()


def mean_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.mean()

In [37]:
numeric_cols = [
    "text_reviews_count",
    "average_rating",
    "ratings_count",
    "num_pages",
    "publication_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [38]:
agg_dict = {
    # Main identity fields
    "clean_title": first_non_missing,
    "authors": first_non_missing,
    "series": unique_list,
    "language_code_final": first_non_missing,

    # IDs for API calls / joins
    "isbn": unique_list,
    "isbn13": unique_list,
    "book_id": unique_list,

    # Edition-level metadata
    "publisher": unique_list,
    "publication_year": unique_list,

    # Book-level text/features
    "description": longest_text,
    "popular_shelves": first_non_missing,
    "similar_books": unique_list,

    # Numeric summary features
    "text_reviews_count": max_numeric,
    "ratings_count": max_numeric,
    "average_rating": mean_numeric,
    "num_pages": max_numeric,
}

# keep only columns that exist in df
agg_dict = {col: func for col, func in agg_dict.items() if col in df.columns}

books_work_df = (
    df.groupby("work_id", dropna=False)
    .agg(agg_dict)
    .reset_index()
)

In [39]:
rename_dict = {
    "series": "series_list",
    "isbn": "isbn_list",
    "isbn13": "isbn13_list",
    "book_id": "book_id_list",
    "publisher": "publisher_list",
    "publication_year": "publication_year_list",
    "similar_books": "similar_books_list",
}

books_work_df = books_work_df.rename(
    columns={k: v for k, v in rename_dict.items() if k in books_work_df.columns}
)

In [40]:
books_work_df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,62,Just Like Tomorrow,"[{'author_id': '883277', 'role': ''}]",[],eng,"[1862301581, 0156030489]","[9781862301580, 9780156030489]","[3353265, 3402]","[Definitions (Young Adult), Mariner Books]",[2006.0],He thought I'd forged my mom's name on the sli...,"[{'count': '191', 'name': 'to-read'}, {'count'...","[[8722298, 13238271, 17739999, 547484, 1213991...",119,888,3.33,192.0
1,388,"Queen Kat, Carmel and St. Jude Get a Life","[{'author_id': '40136', 'role': ''}]",[],eng,"[014028124X, 1742379516]","[9780140281248, 9781742379517]","[71009, 18595773]","[Penguin Global, Allen Unwin]","[2004.0, 2012.0]",Three very different girls from the same count...,"[{'count': '400', 'name': 'to-read'}, {'count'...","[[82588, 1473470, 1144532, 3628128, 1044307, 7...",23,885,3.73,448.0
2,969,"The Prophet of Yonwood (The Ember Series, #3)","[{'author_id': '2347', 'role': ''}]",[[144406]],eng,"[0375875263, 0375840702]","[9780375875267, 9780375840708]","[8486, 7056494]",[Random House Books for Young Readers],[2006.0],It's 50 years before the settlement of the cit...,"[{'count': '11881', 'name': 'to-read'}, {'coun...","[[1084897, 133772, 5953211, 13237056, 569794, ...",81,575,3.26,304.0
3,971,The Stranger,"[{'author_id': '9059', 'role': ''}]",[[1099023]],eng,[0590456806],[9780590456807],[21194],[Scholastic],[1993.0],Drawn to Jethro from the moment she first sees...,"[{'count': '439', 'name': 'to-read'}, {'count'...","[[679312, 601548, 1156197, 309194, 335693, 164...",10,53,3.56,198.0
4,1131,Sleeping Dogs,"[{'author_id': '138742', 'role': ''}]",[],eng,"[0143574019, 0670860042]","[9780143574019, 9780670860043]","[29456147, 6397985]","[Penguin Books Australia, Viking]","[2016.0, 1995.0]","'We must be ruthless,' Edward snarls, 'because...","[{'count': '413', 'name': 'to-read'}, {'count'...","[[6547697, 6443605, 10920730, 16282967, 249405...",2,9,3.78,208.0


In [41]:
df = books_work_df.copy()

In [42]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,62,Just Like Tomorrow,"[{'author_id': '883277', 'role': ''}]",[],eng,"[1862301581, 0156030489]","[9781862301580, 9780156030489]","[3353265, 3402]","[Definitions (Young Adult), Mariner Books]",[2006.0],He thought I'd forged my mom's name on the sli...,"[{'count': '191', 'name': 'to-read'}, {'count'...","[[8722298, 13238271, 17739999, 547484, 1213991...",119,888,3.33,192.0
1,388,"Queen Kat, Carmel and St. Jude Get a Life","[{'author_id': '40136', 'role': ''}]",[],eng,"[014028124X, 1742379516]","[9780140281248, 9781742379517]","[71009, 18595773]","[Penguin Global, Allen Unwin]","[2004.0, 2012.0]",Three very different girls from the same count...,"[{'count': '400', 'name': 'to-read'}, {'count'...","[[82588, 1473470, 1144532, 3628128, 1044307, 7...",23,885,3.73,448.0
2,969,"The Prophet of Yonwood (The Ember Series, #3)","[{'author_id': '2347', 'role': ''}]",[[144406]],eng,"[0375875263, 0375840702]","[9780375875267, 9780375840708]","[8486, 7056494]",[Random House Books for Young Readers],[2006.0],It's 50 years before the settlement of the cit...,"[{'count': '11881', 'name': 'to-read'}, {'coun...","[[1084897, 133772, 5953211, 13237056, 569794, ...",81,575,3.26,304.0
3,971,The Stranger,"[{'author_id': '9059', 'role': ''}]",[[1099023]],eng,[0590456806],[9780590456807],[21194],[Scholastic],[1993.0],Drawn to Jethro from the moment she first sees...,"[{'count': '439', 'name': 'to-read'}, {'count'...","[[679312, 601548, 1156197, 309194, 335693, 164...",10,53,3.56,198.0
4,1131,Sleeping Dogs,"[{'author_id': '138742', 'role': ''}]",[],eng,"[0143574019, 0670860042]","[9780143574019, 9780670860043]","[29456147, 6397985]","[Penguin Books Australia, Viking]","[2016.0, 1995.0]","'We must be ruthless,' Edward snarls, 'because...","[{'count': '413', 'name': 'to-read'}, {'count'...","[[6547697, 6443605, 10920730, 16282967, 249405...",2,9,3.78,208.0


In [43]:
df.shape

(27708, 17)

In [44]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description               527
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                2256
dtype: int64

In [45]:
df.shape

(27708, 17)

In [46]:
df = df[df["description"].notna()].copy()

In [47]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description                 0
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                2131
dtype: int64

In [48]:
df.shape

(27181, 17)

In [49]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,62,Just Like Tomorrow,"[{'author_id': '883277', 'role': ''}]",[],eng,"[1862301581, 0156030489]","[9781862301580, 9780156030489]","[3353265, 3402]","[Definitions (Young Adult), Mariner Books]",[2006.0],He thought I'd forged my mom's name on the sli...,"[{'count': '191', 'name': 'to-read'}, {'count'...","[[8722298, 13238271, 17739999, 547484, 1213991...",119,888,3.33,192.0
1,388,"Queen Kat, Carmel and St. Jude Get a Life","[{'author_id': '40136', 'role': ''}]",[],eng,"[014028124X, 1742379516]","[9780140281248, 9781742379517]","[71009, 18595773]","[Penguin Global, Allen Unwin]","[2004.0, 2012.0]",Three very different girls from the same count...,"[{'count': '400', 'name': 'to-read'}, {'count'...","[[82588, 1473470, 1144532, 3628128, 1044307, 7...",23,885,3.73,448.0
2,969,"The Prophet of Yonwood (The Ember Series, #3)","[{'author_id': '2347', 'role': ''}]",[[144406]],eng,"[0375875263, 0375840702]","[9780375875267, 9780375840708]","[8486, 7056494]",[Random House Books for Young Readers],[2006.0],It's 50 years before the settlement of the cit...,"[{'count': '11881', 'name': 'to-read'}, {'coun...","[[1084897, 133772, 5953211, 13237056, 569794, ...",81,575,3.26,304.0
3,971,The Stranger,"[{'author_id': '9059', 'role': ''}]",[[1099023]],eng,[0590456806],[9780590456807],[21194],[Scholastic],[1993.0],Drawn to Jethro from the moment she first sees...,"[{'count': '439', 'name': 'to-read'}, {'count'...","[[679312, 601548, 1156197, 309194, 335693, 164...",10,53,3.56,198.0
4,1131,Sleeping Dogs,"[{'author_id': '138742', 'role': ''}]",[],eng,"[0143574019, 0670860042]","[9780143574019, 9780670860043]","[29456147, 6397985]","[Penguin Books Australia, Viking]","[2016.0, 1995.0]","'We must be ruthless,' Edward snarls, 'because...","[{'count': '413', 'name': 'to-read'}, {'count'...","[[6547697, 6443605, 10920730, 16282967, 249405...",2,9,3.78,208.0


In [50]:
df = df.drop(columns=["language_code_final", "publication_year_list"])

In [51]:
df.shape

(27181, 15)

In [52]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,62,Just Like Tomorrow,"[{'author_id': '883277', 'role': ''}]",[],"[1862301581, 0156030489]","[9781862301580, 9780156030489]","[3353265, 3402]","[Definitions (Young Adult), Mariner Books]",He thought I'd forged my mom's name on the sli...,"[{'count': '191', 'name': 'to-read'}, {'count'...","[[8722298, 13238271, 17739999, 547484, 1213991...",119,888,3.33,192.0
1,388,"Queen Kat, Carmel and St. Jude Get a Life","[{'author_id': '40136', 'role': ''}]",[],"[014028124X, 1742379516]","[9780140281248, 9781742379517]","[71009, 18595773]","[Penguin Global, Allen Unwin]",Three very different girls from the same count...,"[{'count': '400', 'name': 'to-read'}, {'count'...","[[82588, 1473470, 1144532, 3628128, 1044307, 7...",23,885,3.73,448.0
2,969,"The Prophet of Yonwood (The Ember Series, #3)","[{'author_id': '2347', 'role': ''}]",[[144406]],"[0375875263, 0375840702]","[9780375875267, 9780375840708]","[8486, 7056494]",[Random House Books for Young Readers],It's 50 years before the settlement of the cit...,"[{'count': '11881', 'name': 'to-read'}, {'coun...","[[1084897, 133772, 5953211, 13237056, 569794, ...",81,575,3.26,304.0
3,971,The Stranger,"[{'author_id': '9059', 'role': ''}]",[[1099023]],[0590456806],[9780590456807],[21194],[Scholastic],Drawn to Jethro from the moment she first sees...,"[{'count': '439', 'name': 'to-read'}, {'count'...","[[679312, 601548, 1156197, 309194, 335693, 164...",10,53,3.56,198.0
4,1131,Sleeping Dogs,"[{'author_id': '138742', 'role': ''}]",[],"[0143574019, 0670860042]","[9780143574019, 9780670860043]","[29456147, 6397985]","[Penguin Books Australia, Viking]","'We must be ruthless,' Edward snarls, 'because...","[{'count': '413', 'name': 'to-read'}, {'count'...","[[6547697, 6443605, 10920730, 16282967, 249405...",2,9,3.78,208.0


In [53]:
df["clean_title"] = (
    df["clean_title"]
    .astype(str)
    .str.replace(r"\s*\([^)]*#\d+[^)]*\)", "", regex=True)
    .str.strip()
)

In [54]:
def extract_author_ids(authors):
    if not isinstance(authors, list):
        return []

    return [
        str(a.get("author_id"))
        for a in authors
        if isinstance(a, dict) and a.get("author_id")
    ]

df["author_ids"] = df["authors"].apply(extract_author_ids)

In [55]:
def flatten_nested_list(x):
    if not isinstance(x, list):
        return []

    flat = []

    for item in x:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)

    # remove missing/empty values and dedupe
    result = []
    seen = set()

    for item in flat:
        if item is None:
            continue

        item_str = str(item).strip()

        if item_str == "" or item_str.lower() in {"nan", "none"}:
            continue

        if item_str not in seen:
            seen.add(item_str)
            result.append(item)

    return result

In [56]:
for col in ["series_list", "similar_books_list"]:
    if col in df.columns:
        df[col] = df[col].apply(flatten_nested_list)

In [57]:
numeric_cols = [
    "text_reviews_count",
    "ratings_count",
    "average_rating",
    "num_pages"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [58]:
df["num_pages_missing"] = df["num_pages"].isna().astype(int)

In [59]:
for i, shelves in enumerate(df["popular_shelves"].head(10), start=1):
    print(f"\nRow {i}:")
    print(shelves)


Row 1:
[{'count': '191', 'name': 'to-read'}, {'count': '53', 'name': 'fiction'}, {'count': '46', 'name': 'young-adult'}, {'count': '44', 'name': 'currently-reading'}, {'count': '40', 'name': 'french'}, {'count': '35', 'name': 'france'}, {'count': '30', 'name': 'ya'}, {'count': '13', 'name': 'contemporary'}, {'count': '10', 'name': 'favorites'}, {'count': '9', 'name': 'owned'}, {'count': '8', 'name': 'literature'}, {'count': '8', 'name': 'read-in-french'}, {'count': '8', 'name': 'français'}, {'count': '7', 'name': 'library'}, {'count': '7', 'name': 'french-literature'}, {'count': '6', 'name': 'europe'}, {'count': '6', 'name': 'coming-of-age'}, {'count': '6', 'name': 'en-français'}, {'count': '6', 'name': 'to-buy'}, {'count': '6', 'name': 'translated'}, {'count': '5', 'name': 'read-in-2016'}, {'count': '5', 'name': 'in-french'}, {'count': '5', 'name': 'read-for-school'}, {'count': '5', 'name': 'africa'}, {'count': '5', 'name': 'middle-east'}, {'count': '5', 'name': 'for-school'}, {'coun

In [60]:
int_df = df[['popular_shelves']]

In [61]:
int_df.head()

,popular_shelves
0,"[{'count': '191', 'name': 'to-read'}, {'count'..."
1,"[{'count': '400', 'name': 'to-read'}, {'count'..."
2,"[{'count': '11881', 'name': 'to-read'}, {'coun..."
3,"[{'count': '439', 'name': 'to-read'}, {'count'..."
4,"[{'count': '413', 'name': 'to-read'}, {'count'..."


In [62]:
int_df.shape

(27181, 1)

In [63]:
int_df.to_csv('../data/intermediate/young_shelves.csv', index=False)

In [64]:
# TAG NORMALIZATION MAP ---> BUILT WITH CLAUDE
# instead of calling Claude or gpt's api or even using a model from HuggingFace, this seemed to be the fastest and the cheapest way
shelves = pd.read_csv('../data/intermediate/young_tag_mapping.csv')

In [65]:
shelves.head(20)

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,65293573,27042
1,young-adult,young-adult,keep,2332436,26211
2,currently-reading,status,drop,1846361,23158
3,favorites,review,drop,1616462,16956
4,fantasy,fantasy,keep,1256684,11090
5,ya,young-adult,keep,1182138,21749
6,romance,romance,keep,683381,14601
7,books-i-own,ownership,drop,621821,16485
8,fiction,fiction,keep,539087,18855
9,series,noise,drop,445454,11320


In [66]:
vocab_df = shelves.copy()

In [67]:
tag_map = dict(
    zip(
        vocab_df.loc[vocab_df["action"] == "keep", "raw_tag"],
        vocab_df.loc[vocab_df["action"] == "keep", "clean_tag"]
    )
)

In [68]:
vocab_df.head()

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,65293573,27042
1,young-adult,young-adult,keep,2332436,26211
2,currently-reading,status,drop,1846361,23158
3,favorites,review,drop,1616462,16956
4,fantasy,fantasy,keep,1256684,11090


In [69]:
def replace_shelf_tags(shelves):
    # if shelves got saved as a string, convert it back to list
    if isinstance(shelves, str):
        try:
            shelves = ast.literal_eval(shelves)
        except Exception:
            return []

    if not isinstance(shelves, list):
        return []

    cleaned_shelves = []

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        raw_tag = shelf.get("name")

        # only replace if action == keep and raw_tag exists in tag_map
        if raw_tag in tag_map:
            new_shelf = shelf.copy()
            new_shelf["name"] = tag_map[raw_tag]
            cleaned_shelves.append(new_shelf)

    return cleaned_shelves

In [70]:
df["popular_shelves_cleaned"] = df["popular_shelves"].apply(replace_shelf_tags)

In [71]:
df[["popular_shelves", "popular_shelves_cleaned"]].head()

,popular_shelves,popular_shelves_cleaned
0,"[{'count': '191', 'name': 'to-read'}, {'count'...","[{'count': '53', 'name': 'fiction'}, {'count':..."
1,"[{'count': '400', 'name': 'to-read'}, {'count'...","[{'count': '41', 'name': 'young-adult'}, {'cou..."
2,"[{'count': '11881', 'name': 'to-read'}, {'coun...","[{'count': '490', 'name': 'young-adult'}, {'co..."
3,"[{'count': '439', 'name': 'to-read'}, {'count'...","[{'count': '40', 'name': 'young-adult'}, {'cou..."
4,"[{'count': '413', 'name': 'to-read'}, {'count'...","[{'count': '19', 'name': 'young-adult'}, {'cou..."


In [72]:
print("Original popular_shelves:")
print(df.loc[df.index[0], "popular_shelves"])

print("\nCleaned popular_shelves:")
print(df.loc[df.index[0], "popular_shelves_cleaned"])

Original popular_shelves:
[{'count': '191', 'name': 'to-read'}, {'count': '53', 'name': 'fiction'}, {'count': '46', 'name': 'young-adult'}, {'count': '44', 'name': 'currently-reading'}, {'count': '40', 'name': 'french'}, {'count': '35', 'name': 'france'}, {'count': '30', 'name': 'ya'}, {'count': '13', 'name': 'contemporary'}, {'count': '10', 'name': 'favorites'}, {'count': '9', 'name': 'owned'}, {'count': '8', 'name': 'literature'}, {'count': '8', 'name': 'read-in-french'}, {'count': '8', 'name': 'français'}, {'count': '7', 'name': 'library'}, {'count': '7', 'name': 'french-literature'}, {'count': '6', 'name': 'europe'}, {'count': '6', 'name': 'coming-of-age'}, {'count': '6', 'name': 'en-français'}, {'count': '6', 'name': 'to-buy'}, {'count': '6', 'name': 'translated'}, {'count': '5', 'name': 'read-in-2016'}, {'count': '5', 'name': 'in-french'}, {'count': '5', 'name': 'read-for-school'}, {'count': '5', 'name': 'africa'}, {'count': '5', 'name': 'middle-east'}, {'count': '5', 'name': 'fo

In [73]:
def merge_duplicate_shelf_tags(shelves):
    if not isinstance(shelves, list):
        return []

    merged = OrderedDict()

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        name = shelf.get("name")
        count = shelf.get("count", 0)

        if name is None or str(name).strip() == "":
            continue

        name = str(name).strip()

        try:
            count = int(count)
        except Exception:
            count = 0

        if name not in merged:
            merged[name] = count
        else:
            merged[name] += count

    return [
        {"count": str(count), "name": name}
        for name, count in merged.items()
    ]

In [74]:
df["popular_shelves_cleaned"] = df["popular_shelves_cleaned"].apply(
    merge_duplicate_shelf_tags
)

In [75]:
print(df.loc[df.index[1], "popular_shelves_cleaned"])

[{'count': '76', 'name': 'young-adult'}, {'count': '15', 'name': 'fiction'}, {'count': '7', 'name': 'contemporary'}, {'count': '4', 'name': 'new-adult'}, {'count': '9', 'name': 'children'}, {'count': '3', 'name': 'coming-of-age'}, {'count': '3', 'name': 'romance'}, {'count': '1', 'name': 'school'}, {'count': '1', 'name': 'thriller'}, {'count': '1', 'name': 'mental-health'}, {'count': '1', 'name': 'play'}]


In [76]:
for shelf in df.loc[df.index[10], "popular_shelves_cleaned"]:
    print(shelf)

{'count': '407', 'name': 'young-adult'}
{'count': '189', 'name': 'fiction'}
{'count': '396', 'name': 'children'}
{'count': '44', 'name': 'fantasy'}
{'count': '38', 'name': 'middle-grade'}
{'count': '32', 'name': 'classics'}
{'count': '21', 'name': 'realistic-fiction'}
{'count': '24', 'name': 'family'}
{'count': '9', 'name': 'contemporary'}
{'count': '6', 'name': 'coming-of-age'}
{'count': '11', 'name': 'science-fiction'}
{'count': '6', 'name': 'early-reader'}
{'count': '6', 'name': 'american-history'}
{'count': '4', 'name': 'read-aloud'}
{'count': '4', 'name': 'adult'}
{'count': '3', 'name': 'womens-history'}


In [77]:
df.shape

(27181, 18)

In [78]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,author_ids,num_pages_missing,popular_shelves_cleaned
0,62,Just Like Tomorrow,"[{'author_id': '883277', 'role': ''}]",[],"[1862301581, 0156030489]","[9781862301580, 9780156030489]","[3353265, 3402]","[Definitions (Young Adult), Mariner Books]",He thought I'd forged my mom's name on the sli...,"[{'count': '191', 'name': 'to-read'}, {'count'...","[8722298, 13238271, 17739999, 547484, 12139912...",119,888,3.33,192.0,[883277],0,"[{'count': '62', 'name': 'fiction'}, {'count':..."
1,388,"Queen Kat, Carmel and St. Jude Get a Life","[{'author_id': '40136', 'role': ''}]",[],"[014028124X, 1742379516]","[9780140281248, 9781742379517]","[71009, 18595773]","[Penguin Global, Allen Unwin]",Three very different girls from the same count...,"[{'count': '400', 'name': 'to-read'}, {'count'...","[82588, 1473470, 1144532, 3628128, 1044307, 79...",23,885,3.73,448.0,[40136],0,"[{'count': '76', 'name': 'young-adult'}, {'cou..."
2,969,The Prophet of Yonwood,"[{'author_id': '2347', 'role': ''}]",[144406],"[0375875263, 0375840702]","[9780375875267, 9780375840708]","[8486, 7056494]",[Random House Books for Young Readers],It's 50 years before the settlement of the cit...,"[{'count': '11881', 'name': 'to-read'}, {'coun...","[1084897, 133772, 5953211, 13237056, 569794, 5...",81,575,3.26,304.0,[2347],0,"[{'count': '814', 'name': 'young-adult'}, {'co..."
3,971,The Stranger,"[{'author_id': '9059', 'role': ''}]",[1099023],[0590456806],[9780590456807],[21194],[Scholastic],Drawn to Jethro from the moment she first sees...,"[{'count': '439', 'name': 'to-read'}, {'count'...","[679312, 601548, 1156197, 309194, 335693, 1644...",10,53,3.56,198.0,[9059],0,"[{'count': '81', 'name': 'young-adult'}, {'cou..."
4,1131,Sleeping Dogs,"[{'author_id': '138742', 'role': ''}]",[],"[0143574019, 0670860042]","[9780143574019, 9780670860043]","[29456147, 6397985]","[Penguin Books Australia, Viking]","'We must be ruthless,' Edward snarls, 'because...","[{'count': '413', 'name': 'to-read'}, {'count'...","[6547697, 6443605, 10920730, 16282967, 2494056...",2,9,3.78,208.0,[138742],0,"[{'count': '50', 'name': 'young-adult'}, {'cou..."


In [79]:
df.columns

Index(['work_id', 'clean_title', 'authors', 'series_list', 'isbn_list',
       'isbn13_list', 'book_id_list', 'publisher_list', 'description',
       'popular_shelves', 'similar_books_list', 'text_reviews_count',
       'ratings_count', 'average_rating', 'num_pages', 'author_ids',
       'num_pages_missing', 'popular_shelves_cleaned'],
      dtype='str')

In [80]:
df = df[['work_id', 'clean_title', 'author_ids', 'series_list', 'isbn_list', 
         'isbn13_list', 'book_id_list', 'publisher_list', 'description', 
         'similar_books_list', 'text_reviews_count', 'ratings_count', 
         'average_rating', 'num_pages', 'num_pages_missing', 'popular_shelves_cleaned']]

In [81]:
df.head()

,work_id,clean_title,author_ids,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,num_pages_missing,popular_shelves_cleaned
0,62,Just Like Tomorrow,[883277],[],"[1862301581, 0156030489]","[9781862301580, 9780156030489]","[3353265, 3402]","[Definitions (Young Adult), Mariner Books]",He thought I'd forged my mom's name on the sli...,"[8722298, 13238271, 17739999, 547484, 12139912...",119,888,3.33,192.0,0,"[{'count': '62', 'name': 'fiction'}, {'count':..."
1,388,"Queen Kat, Carmel and St. Jude Get a Life",[40136],[],"[014028124X, 1742379516]","[9780140281248, 9781742379517]","[71009, 18595773]","[Penguin Global, Allen Unwin]",Three very different girls from the same count...,"[82588, 1473470, 1144532, 3628128, 1044307, 79...",23,885,3.73,448.0,0,"[{'count': '76', 'name': 'young-adult'}, {'cou..."
2,969,The Prophet of Yonwood,[2347],[144406],"[0375875263, 0375840702]","[9780375875267, 9780375840708]","[8486, 7056494]",[Random House Books for Young Readers],It's 50 years before the settlement of the cit...,"[1084897, 133772, 5953211, 13237056, 569794, 5...",81,575,3.26,304.0,0,"[{'count': '814', 'name': 'young-adult'}, {'co..."
3,971,The Stranger,[9059],[1099023],[0590456806],[9780590456807],[21194],[Scholastic],Drawn to Jethro from the moment she first sees...,"[679312, 601548, 1156197, 309194, 335693, 1644...",10,53,3.56,198.0,0,"[{'count': '81', 'name': 'young-adult'}, {'cou..."
4,1131,Sleeping Dogs,[138742],[],"[0143574019, 0670860042]","[9780143574019, 9780670860043]","[29456147, 6397985]","[Penguin Books Australia, Viking]","'We must be ruthless,' Edward snarls, 'because...","[6547697, 6443605, 10920730, 16282967, 2494056...",2,9,3.78,208.0,0,"[{'count': '50', 'name': 'young-adult'}, {'cou..."


In [82]:
df.shape

(27181, 16)

In [83]:
df.to_csv('../data/intermediate/books-cleaned/young.csv', index=False)